In [1]:
import os
import gc
import json
import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.io as pio
except ImportError:
    raise ImportError("Plotly is not installed. Run: pip install plotly")

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

PLOTLY_TEMPLATE = "plotly_white"
FIGURES_DIR = "figures_plotly"
TABLES_DIR = "tables_output"

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

STORE_DASHBOARD_OBJECTS = True
SAVE_OUTPUT_FILES = True
SHOW_IN_NOTEBOOK = True
FREE_RAW_DATA_AFTER_PREPROCESSING = True

MIN_REASONABLE_SALE_PRICE = 100_000_000
MAX_PLOT_ROWS = 50_000
MAX_MAP_POINTS = 50_000
RANDOM_STATE = 42

DASHBOARD_FIGS = {}
DASHBOARD_TABLES = {}

PLOTLY_CONFIG = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
    "modeBarButtonsToRemove": [
        "select2d",
        "lasso2d"]}
# در قسمت سوال 4 توصیفی اگر تغییر دادیم تابع زیر و MAX_PLOT_ROWS = 50_000 حذف شوند
def sample_for_plot(df, max_rows=MAX_PLOT_ROWS, random_state=RANDOM_STATE):
    if len(df) > max_rows:
        return df.sample(n=max_rows, random_state=random_state).copy()
    return df.copy()

def show_save_fig(fig, name, height=None):
    if height is not None:
        fig.update_layout(height=height)

    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        font=dict(family="Arial", size=13),
        margin=dict(l=40, r=40, t=70, b=50),
        legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="right",x=1),hoverlabel=dict(font_size=12,font_family="Arial"))

    if STORE_DASHBOARD_OBJECTS:
        DASHBOARD_FIGS[name] = fig
    if SAVE_OUTPUT_FILES:
        html_path = os.path.join(FIGURES_DIR, f"{name}.html")
        json_path = os.path.join(FIGURES_DIR, f"{name}.json")
        fig.write_html(html_path,include_plotlyjs="cdn",full_html=True,config=PLOTLY_CONFIG)
        fig.write_json(json_path,pretty=True)
        print(f"Saved figure HTML: {html_path}")
        print(f"Saved figure JSON: {json_path}")
    if SHOW_IN_NOTEBOOK:
        fig.show(config=PLOTLY_CONFIG)

def store_table(df_table, name, index=False):
    if STORE_DASHBOARD_OBJECTS:
        DASHBOARD_TABLES[name] = df_table.copy()
    if SAVE_OUTPUT_FILES:
        csv_path = os.path.join(TABLES_DIR, f"{name}.csv")
        df_table.to_csv(csv_path, index=index, encoding="utf-8-sig")
        print(f"Saved table: {csv_path}")
    if SHOW_IN_NOTEBOOK:
        display(df_table)

In [2]:
df = pd.read_csv("D:/Quera_bootcamp/project/Divar.csv", low_memory=False)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1000000, 61)


,Unnamed: 0,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,rent_to_single,rent_type,price_mode,price_value,credit_mode,credit_value,rent_credit_transform,transformable_price,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,deed_type,has_business_deed,floor,rooms_count,total_floors_count,unit_per_floor,has_balcony,has_elevator,has_warehouse,has_parking,construction_year,is_rebuilt,has_water,has_warm_water_provider,has_electricity,has_gas,has_heating_system,has_cooling_system,has_restroom,has_security_guard,has_barbecue,building_direction,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,NaN,NaN,NaN,سه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,NaN,NaN,مقطوع,8.500000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,3,یک,NaN,NaN,NaN,True,True,True,۱۳۸۴,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,NaN,NaN,NaN,NaN,مقطوع,750000000.0,False,False,750000000.0,NaN,26000000.0,NaN,NaN,132.0,NaN,NaN,3,سه,NaN,NaN,NaN,True,True,True,۱۴۰۱,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,NaN,NaN,NaN,NaN,مقطوع,950000000.0,False,False,950000000.0,NaN,95000000.0,NaN,NaN,90.0,NaN,NaN,4,یک,NaN,NaN,NaN,True,False,True,۱۴۰۰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,NaN,NaN,مقطوع,5.750000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,115.0,single_page,NaN,4,دو,6,NaN,true,True,True,True,۱۴۰۳,NaN,NaN,package,NaN,NaN,shoofaj,air_conditioner,squat_seat,NaN,NaN,north,NaN,NaN,NaN,ceramic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
info_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean().values * 100).round(2),
    "unique_count": df.nunique(dropna=True).values})
info_df = info_df.sort_values("missing_percent", ascending=False)
store_table(info_df, "raw_data_quality_report")

Saved table: tables_output\raw_data_quality_report.csv


,column,dtype,missing_count,missing_percent,unique_count
rent_to_single,rent_to_single,object,999981,100.00,2
cost_per_extra_person,cost_per_extra_person,float64,989759,98.98,218
rent_price_on_special_days,rent_price_on_special_days,float64,989537,98.95,379
rent_price_at_weekends,rent_price_at_weekends,float64,986449,98.64,416
rent_price_on_regular_days,rent_price_on_regular_days,float64,981932,98.19,500
extra_person_capacity,extra_person_capacity,object,975991,97.60,36
property_type,property_type,object,972943,97.29,5
has_sauna,has_sauna,object,971521,97.15,2
has_jacuzzi,has_jacuzzi,object,971272,97.13,2
has_pool,has_pool,object,970610,97.06,2


In [5]:
data = df.copy()
if "Unnamed: 0" in data.columns:
    data = data.drop(columns=["Unnamed: 0"])
print("Shape after dropping Unnamed column:", data.shape)
display(data.head())

Shape after dropping Unnamed column: (1000000, 60)


,cat2_slug,cat3_slug,city_slug,neighborhood_slug,created_at_month,user_type,description,title,rent_mode,rent_value,rent_to_single,rent_type,price_mode,price_value,credit_mode,credit_value,rent_credit_transform,transformable_price,transformable_credit,transformed_credit,transformable_rent,transformed_rent,land_size,building_size,deed_type,has_business_deed,floor,rooms_count,total_floors_count,unit_per_floor,has_balcony,has_elevator,has_warehouse,has_parking,construction_year,is_rebuilt,has_water,has_warm_water_provider,has_electricity,has_gas,has_heating_system,has_cooling_system,has_restroom,has_security_guard,has_barbecue,building_direction,has_pool,has_jacuzzi,has_sauna,floor_material,property_type,regular_person_capacity,extra_person_capacity,cost_per_extra_person,rent_price_on_regular_days,rent_price_on_special_days,rent_price_at_weekends,location_latitude,location_longitude,location_radius
0,temporary-rent,villa,karaj,mehrshahr,2024-08-01 00:00:00,مشاور املاک,۵۰۰متر\n۲۰۰متر بنا دوبلکس\n۳خواب\nاستخر آبگرم ...,باغ ویلا اجاره روزانه استخر داخل لشکرآباد سهیلیه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0,NaN,NaN,NaN,سه,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,6,350000.0,1500000.0,3.500000e+09,3500000.0,35.811684,50.936600,500.0
1,residential-sell,apartment-sell,tehran,gholhak,2024-05-01 00:00:00,مشاور املاک,دسترسی عالی به مترو و شریعتی \nمشاعات تمیز \nب...,۶۰ متر قلهک فول امکانات,NaN,NaN,NaN,NaN,مقطوع,8.500000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,3,یک,NaN,NaN,NaN,True,True,True,۱۳۸۴,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,500.0
2,residential-rent,apartment-rent,tehran,tohid,2024-10-01 00:00:00,NaN,تخلیه پایان ماه,آپارتمان ۳ خوابه ۱۳۲ متر,مقطوع,26000000.0,NaN,NaN,NaN,NaN,مقطوع,750000000.0,False,False,750000000.0,NaN,26000000.0,NaN,NaN,132.0,NaN,NaN,3,سه,NaN,NaN,NaN,True,True,True,۱۴۰۱,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.703865,51.373459,NaN
3,commercial-rent,office-rent,tehran,elahiyeh,2024-06-01 00:00:00,NaN,فرشته تاپ لوکیشن\n۹۰ متر موقعیت اداری\nیک اتاق...,فرشته ۹۰ متر دفتر کار مدرن موقعیت اداری,مقطوع,95000000.0,NaN,NaN,NaN,NaN,مقطوع,950000000.0,False,False,950000000.0,NaN,95000000.0,NaN,NaN,90.0,NaN,NaN,4,یک,NaN,NaN,NaN,True,False,True,۱۴۰۰,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,residential-sell,apartment-sell,mashhad,emamreza,2024-05-01 00:00:00,مشاور املاک,هلدینگ ساختمانی اکبری\n\nهمراه شما هستیم برای ...,۱۱۵ متری/شمالی رو به آفتاب/اکبری,NaN,NaN,NaN,NaN,مقطوع,5.750000e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,115.0,single_page,NaN,4,دو,6,NaN,true,True,True,True,۱۴۰۳,NaN,NaN,package,NaN,NaN,shoofaj,air_conditioner,squat_seat,NaN,NaN,north,NaN,NaN,NaN,ceramic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
persian_arabic_digits = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩","01234567890123456789")
def normalize_digits(x):
    if pd.isna(x):
        return x
    return str(x).translate(persian_arabic_digits)

def safe_numeric(series):
    s = (series.astype(str).map(normalize_digits).str.replace(",", "", regex=False).str.replace("٬", "", regex=False).str.replace("،", "", regex=False).str.strip())
    extracted = s.str.extract(r"(-?\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(extracted, errors="coerce")

def clean_construction_year(series):
    s = (series.astype(str).map(normalize_digits).str.replace(",", "", regex=False).str.replace("٬", "", regex=False).str.replace("،", "", regex=False).str.strip())
    result = pd.Series(np.nan, index=series.index, dtype="float")
    before_1370_mask = s.str.contains("قبل از 1370", na=False)
    result.loc[before_1370_mask] = 1369
    extracted = pd.to_numeric(s.str.extract(r"(\d{4})")[0], errors="coerce")
    result = result.fillna(extracted)
    result = result.where(result.between(1300, 1405), np.nan)
    return result

def clean_rooms(series):
    room_map = {
        "بدون اتاق": 0,
        "بدون": 0,
        "استودیو": 0,
        "سوئیت": 0,
        "یک": 1,
        "۱": 1,
        "1": 1,
        "دو": 2,
        "۲": 2,
        "2": 2,
        "سه": 3,
        "۳": 3,
        "3": 3,
        "چهار": 4,
        "۴": 4,
        "4": 4,
        "پنج": 5,
        "پنج یا بیشتر": 5,
        "۵": 5,
        "5": 5}

    s = (series.astype(str).map(normalize_digits).str.strip().str.lower())
    direct_num = pd.to_numeric(s, errors="coerce")
    mapped = s.map(room_map)
    extracted = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")
    result = direct_num.fillna(mapped).fillna(extracted)
    result = result.where(result.between(0, 10), np.nan)
    return result


numeric_cols = [
    "rent_value",
    "price_value",
    "credit_value",
    #"rent_credit_transform",          این 2 تا ستون بولین هستن
    #"transformable_price",
    "transformable_credit",
    "transformed_credit",
    "transformable_rent",
    "transformed_rent",
    "land_size",
    "building_size",
    "floor",
    "rooms_count",
    "total_floors_count",
    "unit_per_floor",
    "construction_year",
    "regular_person_capacity",
    "extra_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
    "location_latitude",
    "location_longitude",
    "location_radius"]

for col in numeric_cols:
    if col in data.columns:
        if col == "rooms_count":
            data[col] = clean_rooms(data[col])
        elif col == "construction_year":
            data[col] = clean_construction_year(data[col])
        else:
            data[col] = safe_numeric(data[col])

print("Numeric conversion done.")
available_numeric_cols = [col for col in numeric_cols if col in data.columns]
display(data[available_numeric_cols].head())
print("\nrooms_count value counts after cleaning:")
display(data["rooms_count"].value_counts(dropna=False).sort_index())

data["effective_monthly_rent"] = np.nan
rent_mask = data["cat2_slug"].isin(["residential-rent", "commercial-rent"])
data.loc[rent_mask, "effective_monthly_rent"] = (data.loc[rent_mask, "rent_value"].fillna(0) +data.loc[rent_mask, "credit_value"].fillna(0) * 0.03)